# Robustness fine-tune of the five agents (training split only)

Continues training from each released checkpoint with the released architecture, loss and
inference recipe, under deployment-style corruption augmentation, on the **training split
only**. Every selection uses the clean **validation** split; a retrained agent replaces the
released one only if its validation log-loss is lower and its validation AUC-ROC is within
0.002 of the released agent's (`tools/robust_finetune/common.py`). The test split and the
frozen YouTube set are read once afterwards, offline.

Outputs (checkpoints, histories, per-clip val/test score CSVs, decision.json per agent) are
written to Drive so the run survives a disconnect; re-running the driver resumes.
Runtime on a T4: roughly 1.5 to 2.5 h for all five agents plus test scoring.


In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
%cd /content
!rm -rf repo && git clone -q -b robust-finetune https://github.com/saoirsebarry/multiagent-deepfake-detection.git repo
%cd /content/repo
!pip -q install speechbrain timm librosa opencv-python-headless
!git log --oneline -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROCESSED = '/content/drive/MyDrive/polyglotfake/processed'
OUT = '/content/drive/MyDrive/polyglotfake/robust_ft'
os.makedirs(OUT, exist_ok=True)
for s in ('train', 'val', 'test'):
    d = os.path.join(PROCESSED, s); n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'{s:5s} {n:5d} clips')
assert os.path.isdir(os.path.join(PROCESSED, 'train')) and os.path.isdir(os.path.join(PROCESSED, 'val'))

In [ ]:
# Stage the splits on local disk: Drive-backed random reads throttle the data loaders.
import shutil, os, time
LOCAL = '/content/pgf'
for s in ('train', 'val', 'test'):
    src, dst = os.path.join(PROCESSED, s), os.path.join(LOCAL, s)
    if os.path.isdir(src) and not os.path.isdir(dst):
        t = time.time(); shutil.copytree(src, dst); print(f'copied {s} in {time.time()-t:.0f}s')
!du -sh /content/pgf/* 2>/dev/null

In [ ]:
%cd /content/repo
!python -u tools/robust_finetune/run_all.py --data_dir /content/pgf --out_dir "$OUT" 2>&1 | tee "$OUT/run_all.log"
!for a in biometric crossmodal freqnet ecapa xception; do echo "== $a"; tail -n 4 "$OUT/$a/train.log"; done

In [ ]:
import json, glob, os
for p in sorted(glob.glob(os.path.join(OUT, '*', 'decision.json'))):
    d = json.load(open(p)); r, b = d['released_val'], d['best_val']
    print(f"{d['agent']:10s} released logloss {r['logloss']:.4f} auc {r['auc']:.5f} | best logloss {b['logloss']:.4f} auc {b['auc']:.5f} (epoch {b.get('epoch')}) | adopted={d['adopted']}")